# Export Data

In [ ]:
pip install pymysql

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.3/45.3 kB 4.1 MB/s eta 0:00:00


In [ ]:
import os
import shutil
import pymysql
import random
import requests
import urllib.parse
import csv
from google.colab import drive

SRC_DIR = "images"   
OUTPUT_DIR = "dataset_split"
os.makedirs(OUTPUT_DIR, exist_ok=True)

Define host, port, user, password, and database name in .env.



In [ ]:
try:
    conn = pymysql.connect(
        host=db_host,
        port=db_port,
        user=db_user,
        password=db_password,
        database=db_name,
        cursorclass=pymysql.cursors.DictCursor,
        charset="utf8mb4",
        use_unicode=True,
    )
    print("Attempted to connect to MySQL database.")
except pymysql.Error as e:
    print(f"Error: {e}")

Attempted to connect to MySQL database.


In [ ]:
try:
    cursor = conn.cursor()
    print("Cursor created successfully.")
except pymysql.Error as e:
    print(f"Err: {e}")

Cursor created successfully.


In [ ]:
try:
    sql_query = "SELECT * FROM Records WHERE SourceType='market' AND Number!='' AND Number NOT LIKE '%\-%' AND ImagesSources!=''  ORDER BY RAND() LIMIT 5000"
    cursor.execute(sql_query)
    print("SQL query executed successfully on 'Records' table.")

    records = cursor.fetchall()
    print(f"Fetched {len(records)} records.")

except pymysql.Error as e:
    print(f"Error: {e}")

<>:3: SyntaxWarning: invalid escape sequence '\-'
<>:3: SyntaxWarning: invalid escape sequence '\-'
/tmp/ipython-input-2240308593.py:3: SyntaxWarning: invalid escape sequence '\-'
  sql_query = "SELECT * FROM Records WHERE SourceType='market' AND Number!='' AND Number NOT LIKE '%\-%' AND ImagesSources!=''  ORDER BY RAND() LIMIT 5000"


SQL query executed successfully on 'Records' table.
Fetched 5000 records.


In [ ]:
print(f"Displaying the {len(records)} fetched records:")
for i, record in enumerate(records):
    if i >= 500:
        break
    print(record['Id'],record['MakeId'], record['ModelId'], record['ColorId'], record['SourceUrl'], record['VehicleVIN'], record['Number'], record['ImagesSources'].split(';')[0])

Displaying the 5000 fetched records:
36783593 34 5587 12 https://auto.ria.com/auto_kia_ceed_37709293.html Y6DHC512BBL254744 AA3165TB https://img.youtube.com/vi/el9-Yu3RegA/default.jpg
43225406 42 6407018 1 https://auto.ria.com/auto_bmw_3_series_39055743.html WBAUX310X0A710804 AM9193HI https://cdn4.riastatic.com/photosnew/auto/photo/bmw_3-series__619587469f.jpg
30530882 43 2609911 12 https://auto.ria.com/auto_lexus_nx_36299469.html JTJBJRBZ402039505 AP9494EI https://cdn4.riastatic.com/photosnew/auto/photo/lexus_nx__543580874f.jpg
35780946 47 2537 12 https://auto.ria.com/auto_nissan_x_trail_37426840.html JN1TBNT30U0005383 CB9195EK https://cdn3.riastatic.com/photosnew/auto/photo/nissan_x-trail__574652428f.jpg
40834488 10 1245 None https://auto.ria.com/auto_mitsubishi_outlander_38301667.html JA4AZ3A32KZ050654 CE5363CM https://cdn1.riastatic.com/photosnew/auto/photo/mitsubishi_outlander__598935146f.jpg
32288039 7 6408893 12 https://auto.ria.com/auto_mercedes_benz_c_class_36505767.html WDB20

In [ ]:
color_map = {
    1: "white",
    2: "beige",
    3: "yellow",
    4: "green",
    5: "brown",
    6: "orange",
    7: "gray",
    8: "blue",
    9: "purple",
    10: "red",
    11: "black",
    12: "unknown",
    13: "burgundy",
    14: "burn",
    15: "charcoal",
    16: "cream",
    17: "gold",
    18: "maroon",
    19: "pink",
    20: "silver",
    21: "tan",
    22: "teal",
    23: "turquoise",
    24: "champagne",
    25: "dark blue",
    26: "dark brown",
    27: "light blue",
    28: "navy",
    29: "pewter"
}

for r in records:
    cid = r.get("ColorId")
    if isinstance(cid, int) and cid in color_map:

        r["color_name"] = color_map[cid]
    else:
        r["color_name"] = "unknown"

In [ ]:
print(f"Displaying the {len(records)} fetched records:")
for i, record in enumerate(records):
    if i >= 500:
        break
    print(record['Id'],record['MakeId'], record['ModelId'], record['color_name'], record['SourceUrl'], record['VehicleVIN'], record['Number'], record['ImagesSources'].split(';')[0])

Displaying the 5000 fetched records:
36783593 34 5587 unknown https://auto.ria.com/auto_kia_ceed_37709293.html Y6DHC512BBL254744 AA3165TB https://img.youtube.com/vi/el9-Yu3RegA/default.jpg
43225406 42 6407018 white https://auto.ria.com/auto_bmw_3_series_39055743.html WBAUX310X0A710804 AM9193HI https://cdn4.riastatic.com/photosnew/auto/photo/bmw_3-series__619587469f.jpg
30530882 43 2609911 unknown https://auto.ria.com/auto_lexus_nx_36299469.html JTJBJRBZ402039505 AP9494EI https://cdn4.riastatic.com/photosnew/auto/photo/lexus_nx__543580874f.jpg
35780946 47 2537 unknown https://auto.ria.com/auto_nissan_x_trail_37426840.html JN1TBNT30U0005383 CB9195EK https://cdn3.riastatic.com/photosnew/auto/photo/nissan_x-trail__574652428f.jpg
40834488 10 1245 unknown https://auto.ria.com/auto_mitsubishi_outlander_38301667.html JA4AZ3A32KZ050654 CE5363CM https://cdn1.riastatic.com/photosnew/auto/photo/mitsubishi_outlander__598935146f.jpg
32288039 7 6408893 unknown https://auto.ria.com/auto_mercedes_benz_

In [ ]:
cursor.execute("SELECT Id, Name FROM Makes")
makes_rows = cursor.fetchall()
make_map = {row["Id"]: row["Name"] for row in makes_rows}
cursor.execute("SELECT Id, Name FROM Models")
models_rows = cursor.fetchall()

model_map = {row["Id"]: row["Name"] for row in models_rows}


In [ ]:
for r in records:
    make_id = r.get("MakeId")
    model_id = r.get("ModelId")

    if isinstance(make_id, int) and make_id in make_map:
        r["make_name"] = make_map[make_id]
    else:
        r["make_name"] = "unknown"

    if isinstance(model_id, int) and model_id in model_map:
        r["model_name"] = model_map[model_id]
    else:
        r["model_name"] = "unknown"

In [ ]:
print(f"Displaying the {len(records)} fetched records:")
for i, record in enumerate(records):
    if i >= 50:
        break
    print(record['Id'],record['make_name'], record['model_name'],record['Year'], record['color_name'], record['SourceUrl'], record['VehicleVIN'], record['Number'], record['ImagesSources'].split(';')[0])

Displaying the 5000 fetched records:
36783593 kia ceed 2010 unknown https://auto.ria.com/auto_kia_ceed_37709293.html Y6DHC512BBL254744 AA3165TB https://img.youtube.com/vi/el9-Yu3RegA/default.jpg
43225406 bmw 3 series 2010 white https://auto.ria.com/auto_bmw_3_series_39055743.html WBAUX310X0A710804 AM9193HI https://cdn4.riastatic.com/photosnew/auto/photo/bmw_3-series__619587469f.jpg
30530882 lexus nx 300h 2016 unknown https://auto.ria.com/auto_lexus_nx_36299469.html JTJBJRBZ402039505 AP9494EI https://cdn4.riastatic.com/photosnew/auto/photo/lexus_nx__543580874f.jpg
35780946 nissan x-trail 2003 unknown https://auto.ria.com/auto_nissan_x_trail_37426840.html JN1TBNT30U0005383 CB9195EK https://cdn3.riastatic.com/photosnew/auto/photo/nissan_x-trail__574652428f.jpg
40834488 mitsubishi outlander 2019 unknown https://auto.ria.com/auto_mitsubishi_outlander_38301667.html JA4AZ3A32KZ050654 CE5363CM https://cdn1.riastatic.com/photosnew/auto/photo/mitsubishi_outlander__598935146f.jpg
32288039 mercede

In [ ]:
fieldnames = [
    "id",
    "make_name",
    "model_name",
    "year",
    "color_name",
    "source_url",
    "number",
    "image_url",    
]

output_file = "vehicles.csv"

rows = []

for r in records:
    image_url = ""
    images_sources = r.get("ImagesSources") or ""
    if images_sources:
        image_url = images_sources.split(";")[0]

    row = {
        "id": r.get("Id"),
        "make_name": r.get("make_name"),
        "model_name": r.get("model_name"),
        "year": r.get("Year"),
        "color_name": r.get("color_name"),
        "source_url": r.get("SourceUrl"),
        # "vehicle_vin": r.get("VehicleVIN"),
        "number": r.get("Number"),
        "image_url": image_url,
    }
    rows.append(row)

with open(output_file, "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(rows)

Экспортировано 5000 записей в vehicles.csv


In [ ]:
seen_urls = set()
success = 0
failed = 0

for row in rows:
    rec_id = row.get("id")  
    if rec_id is None:
        continue 

    url = row.get("image_url") or ""
    if url in seen_urls:
        continue
    seen_urls.add(url)

    try:
        resp = requests.get(url, timeout=10)
        resp.raise_for_status()
    except Exception as e:
        print(f"[FAIL] {rec_id}: {url} — {e}")
        failed += 1
        continue

    content_type = (resp.headers.get("Content-Type") or "").lower()
    ext = ""
    if content_type.startswith("image/"):
        ext = "." + content_type.split("/")[-1].split(";")[0]

    if not ext or len(ext) > 10:
        path = urllib.parse.urlparse(url).path
        _, ext = os.path.splitext(path)

    if not ext:
        ext = ".jpg"

    base_name = f"{rec_id}"
    safe_base = "".join(
        c if c.isalnum() or c in ("-", "_") else "_"
        for c in str(base_name)
    )
    filename = safe_base + ext
    file_path = os.path.join(OUTPUT_DIR, filename)

    try:
        with open(file_path, "wb") as f:
            f.write(resp.content)
        success += 1
        print(f"[OK] {rec_id}: {url} -> {file_path}")
    except Exception as e:
        print(f"[FAIL WRITE] {file_path} — {e}")
        failed += 1

print(f"\nГотово! Успешно: {success}, с ошибкой: {failed}")


Streaming output truncated to the last 5000 lines.
[OK] 30530882: https://cdn4.riastatic.com/photosnew/auto/photo/lexus_nx__543580874f.jpg -> images/30530882.jpeg
[OK] 35780946: https://cdn3.riastatic.com/photosnew/auto/photo/nissan_x-trail__574652428f.jpg -> images/35780946.jpeg
[OK] 40834488: https://cdn1.riastatic.com/photosnew/auto/photo/mitsubishi_outlander__598935146f.jpg -> images/40834488.jpeg
[OK] 32288039: https://cdn3.riastatic.com/photosnew/auto/photo/mercedes-benz_c-class__549199173f.jpg -> images/32288039.jpeg
[OK] 35077542: https://cdn2.riastatic.com/photosnew/auto/photo/mitsubishi_outlander__565924282f.jpg -> images/35077542.jpeg
[OK] 35905401: https://cdn3.riastatic.com/photosnew/auto/photo/mitsubishi_outlander__575278123f.jpg -> images/35905401.jpeg
[OK] 32901794: https://cdn4.riastatic.com/photosnew/auto/photo/mazda_cx-7__551531744f.jpg -> images/32901794.jpeg
[OK] 32282161: https://cdn0.riastatic.com/photosnew/auto/photo/toyota_camry__549431295f.jpg -> images/322821

In [ ]:
OUTPUT_DIR = "images"
os.makedirs(OUTPUT_DIR, exist_ok=True)
shutil.make_archive("images_archive", "zip", "images")

'/content/images_archive.zip'

In [ ]:
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
all_files = [f for f in os.listdir(SRC_DIR)
             if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
random.shuffle(all_files) 

part1 = all_files[:2500]
part2 = all_files[2500:]

splits = {
    "train": 1500,
    "val": 500,
    "test": 500,
}

def split_and_copy(files, base_out_dir):
    os.makedirs(base_out_dir, exist_ok=True)

    test = files[:splits["test"]]
    val = files[splits["test"]: splits["test"] + splits["val"]]
    train = files[splits["test"] + splits["val"]:]  # остаток

    mapping = {
        "train": train,
        "val": val,
        "test": test,
    }

    for split_name, flist in mapping.items():
        out_dir = os.path.join(base_out_dir, split_name)
        os.makedirs(out_dir, exist_ok=True)

        for fname in flist:
            src_path = os.path.join(SRC_DIR, fname)
            dst_path = os.path.join(out_dir, fname)
            shutil.copy(src_path, dst_path)

    print(f"Готово: {base_out_dir}")
    print(f"train: {len(train)}, val: {len(val)}, test: {len(test)}")

split_and_copy(part1, os.path.join(OUTPUT_DIR, "part1"))
split_and_copy(part2, os.path.join(OUTPUT_DIR, "part2"))


Готово: dataset_split/part1
train: 1500, val: 500, test: 500
Готово: dataset_split/part2
train: 1489, val: 500, test: 500


In [ ]:
shutil.make_archive("dataset_split_2", "zip", "dataset_split")

'/content/dataset_split_2.zip'